In [ ]:
#Creates a GUI using tkinter that allows the user to search for a player's stats in a given time period. 

#Debugged and aided in coding using Open AI's ChatGPT

"""Author: OpenAI
Year: 2023
Title: ChatGPT
URL: https://www.openai.com/research/chatgpt"""

#Used RealPython.com to learn how to create a GUI using tkinter
"""Author: David Amos
Title: Python GUI Programming With Tkinter
URL: https://realpython.com/python-gui-tkinter/"""


import tkinter as tk
import pandas as pd
import chardet


# function created to find header rows based on a specific keyword in the csv file
def find_header_rows(file_path, encoding):
    with open(file_path, 'r', encoding=encoding) as file:
        header_rows = []
        for i, line in enumerate(file):
            if 'Player' in line:
                header_rows.append(i+1)
    return header_rows

def clear_results():
    # Clears the result text box 
    result_text.config(state=tk.NORMAL)
    result_text.delete(1.0, tk.END)
    result_text.config(state=tk.DISABLED)

    # Reset entry fields
    entry_start.delete(0, tk.END)
    entry_end.delete(0, tk.END)
    entry_player_name.delete(0, tk.END)
    entry_selected_stats.delete(0, tk.END)

# function to fetch players stats based on the parameters inputted by the user
def get_player_stats():
    # gets input values from the entry fields
    season_range_start = entry_start.get()
    season_range_end = entry_end.get()
    player_name = entry_player_name.get()
    selected_stats = entry_selected_stats.get()

    # iterates through seasons and gets stats for the specified player
    for season in range(int(season_range_start), int(season_range_end) + 1):
        file_path = fr'C:\Users\febur\Downloads\Capstone Project Data\TEST{season}.csv'
        player_stats = fetch_player_stats(file_path, player_name, selected_stats)
        formatted_stats = format_stats(player_stats)
        display_stats(f"Stats for {player_name} in {season-1}-{season}:", formatted_stats)

# function that detects encoding, reads the csv file, and retrieves players stats
def fetch_player_stats(data_file, player_name, selected_stats=None):
    with open(data_file, 'rb') as f:
        result = chardet.detect(f.read())
    encoding = result['encoding']
    header_rows = find_header_rows(data_file, encoding)
    df = pd.read_csv(data_file, header=1, encoding=encoding)
    
    # Convert player_name to lowercase for comparison
    player_name_lower = player_name.lower()
    
    # Store original 'Player' column for case-sensitive output
    original_player_column = df['Player']
    
    # Convert 'Player' column values to lowercase for comparison
    df['Player'] = df['Player'].str.lower()
    
    # filters players stats based on player_name
    player_stats = df[df['Player'] == player_name_lower]
    
    if player_stats.empty:
        return f"No stats found for {player_name}"
    else:
        # Get original case 'Player' values for output
        player_stats.loc[:, 'Player'] = original_player_column[player_stats.index].values
        
        if selected_stats is None or selected_stats.lower() == 'all':
            return player_stats.to_string(index=False)
        else:
            selected_stats = [stat.strip() for stat in selected_stats.split(',')]
            return player_stats[selected_stats].to_string(index=False)

# function that formats the retrieved player's stats for display
def format_stats(player_stats):
    if player_stats.startswith("No stats found"):
        return player_stats  # No stats available

    stats_list = player_stats.split('\n') # creates new line for each stat

    headers = stats_list[0].split()  # Assuming the first line contains headers

    values = stats_list[1].split()

    # Combining 'Player' value if it's split into two words, first and last name
    combined_player = ' '.join(values[1:3]) if len(values) > 2 else values[0]
    player_name = [values[0]]  # Keep the first value as a separate entity
    values = player_name + [combined_player] + values[3:]  # Replace the second and third values with combined 'Player'
    formatted_stats = '\n'.join(f"{header:0}: {value}" for header, value in zip(headers, values))
    return formatted_stats


def display_stats(title, stats):
    result_text.config(state=tk.NORMAL)
    result_text.insert(tk.END, f"{title}\n{stats}\n\n")
    result_text.config(state=tk.DISABLED)


# sets up GUI using tkinter
root = tk.Tk()
root.title("Player Stats Search")

# labels entry fields for user input
label_season_start = tk.Label(root, text="Season Start:")
label_season_start.pack()

entry_start = tk.Entry(root)
entry_start.pack()

label_season_end = tk.Label(root, text="Season End:")
label_season_end.pack()

entry_end = tk.Entry(root)
entry_end.pack()

label_player_name = tk.Label(root, text="Player's Name:")
label_player_name.pack()

entry_player_name = tk.Entry(root)
entry_player_name.pack()

label_selected_stats = tk.Label(root, text="Selected Stats:")
label_selected_stats.pack()

entry_selected_stats = tk.Entry(root)
entry_selected_stats.pack()

# creates button for user action
search_button = tk.Button(root, text="Search Stats", command=get_player_stats)
search_button.pack()

result_text = tk.Text(root, height=15, width=50)
result_text.config(state=tk.DISABLED)
result_text.pack()

new_search_button = tk.Button(root, text="New Search", command=clear_results)
new_search_button.pack()

root.mainloop()